In [78]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
import function_sampling_Thermal_forcing_files as fn
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress
from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'
variable2 = 'iareafl'

In [79]:
 
mnt_pth = '/Volumes/CrucialX8/computing/data/antarctica/' #mount path
pth_imip6hack= mnt_pth + 'ismip6_hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6_hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6_hackathon/ComputedScalarsHelene/'
 
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = ['IMAU_UFEMISM1', 'IMAU_UFEMISM2', 'IMAU_UFEMISM3', 'IMAU_UFEMISM4',
               'DOE_MALI_4km', 'DOE_MALI_8km_Ant95', 'DOE_MALI_8km_AntMean'] # specify models to remove
 
metadata_file = 'Metadata.txt'
# Generate loop_info DataFrame

In [80]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]

In [81]:
dic_area_of_interest = {}
dic_area_of_interest['FilchnerRonne'] = ['_sector_5','_sector_11']
dic_area_of_interest['Ross'] = ['_sector_2','_sector_6']
dic_area_of_interest['Amundsen'] = ['_sector_4']


In [82]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/thermalforcing/computed_thermalforcing_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'


    #thermal foring 
#     d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )

    
    ind = np.where((d.time.values >=2070) & (d.time.values <= 2100))
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d2[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]=  yais
   

    for j in range(1,19):
        y =d2[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        
    for j in range(1,4):
        y =d2[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        for j,sector in enumerate(sectors):
            ys[j,:] = d2[f'{variable2}{sector}'].values[:tmin].copy()
        sumy = np.sum(ys.copy(),axis = 0)
        df[key].iloc[i]= np.nanmean(sumy[ind]).copy()
        
        

        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/{variable2}_2100.csv', index=False)



/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/2363817180.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/2363817180.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = y[ind].mean()
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/2363817180.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-vie

In [76]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/thermalforcing/computed_thermalforcing_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'


    #thermal foring 
#     d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )

    
    ind = np.where((d.time.values >=2170) & (d.time.values <= 2200))
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d2[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]=  yais
   

    for j in range(1,19):
        y =d2[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        
    for j in range(1,4):
        y =d2[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        for j,sector in enumerate(sectors):
            ys[j,:] = d2[f'{variable2}{sector}'].values[:tmin].copy()
        sumy = np.sum(ys.copy(),axis = 0)
        df[key].iloc[i]= np.nanmean(sumy[ind]).copy()
        
        

        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/{variable2}_2200.csv', index=False)


/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/777185114.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/777185114.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = y[ind].mean()
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/777185114.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-v

In [63]:
df

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_15,sector_16,sector_17,sector_18,region_1,region_2,FilchnerRonne,Ross,Amundsen,AIS
0,expAE02,DC_ISSM,/Volumes/CrucialX8/computing/data/antarctica/i...,26208309248,328156479488,26259832832,22817908736,365059866624,234651516928,15723011072,...,90798432256,31774119936,2.349983e+10,22221424640,768502530048,773371592704,5.750019e+11,5.628080e+11,2.281791e+10,1619369656320
1,expAE02,IGE_ElmerIce,/Volumes/CrucialX8/computing/data/antarctica/i...,7965477888,311750721536,19754539008,17412534272,308516552704,234327212032,8893003776,...,36235608064,1925213440,1.580376e+09,10271821824,665399721984,619117084672,4.897472e+11,5.460780e+11,1.741253e+10,1298294243328
2,expAE02,ILTS_SICOPOLIS,/Volumes/CrucialX8/computing/data/antarctica/i...,9000211456,383880855552,13787720704,20843489280,430425702400,237313818624,21111326720,...,81527300096,24202266624,1.805322e+10,6286142464,857938067456,753894490112,6.599313e+11,6.211947e+11,2.084349e+10,1660374220800
3,expAE02,LSCE_GRISLI2,/Volumes/CrucialX8/computing/data/antarctica/i...,9144881152,183743365120,11069212672,4944662016,79312822272,34249181184,6777843200,...,15442131968,6140692480,2.368517e+09,6956079104,288214941696,159454167040,1.168881e+11,2.179925e+11,4.944662e+09,463134359552
4,expAE02,LSCE_GRISLI,/Volumes/CrucialX8/computing/data/antarctica/i...,5140168192,219033714688,5318920704,3866828032,74277167104,23950766080,4235002880,...,12411904000,3666711040,1.032474e+09,4266505216,307636731904,114202476544,9.900766e+10,2.429844e+11,3.866828e+09,430804992000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,expAE05,VUW_PISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,6104545280,84449501184,16870647808,16518554624,103329497088,13350931456,6893416448,...,20499472384,5226190848,8.530474e+09,11797696512,227272736768,242738823168,1.420367e+11,9.780043e+10,1.651856e+10,495565864960
140,expAE05,VUW_PISM2_s1,/Volumes/CrucialX8/computing/data/antarctica/i...,7449758208,90810793984,19615975424,16844766208,97609064448,13220431872,9621811200,...,21675065344,4672238592,9.585027e+09,12838450176,232330379264,246820159488,1.363501e+11,1.040312e+11,1.684477e+10,506246266880
141,expAE05,VUW_PISM2_s2,/Volumes/CrucialX8/computing/data/antarctica/i...,5915857920,79341142016,16238454784,17434615808,102961889280,13133936640,6162634752,...,19531264000,5285320704,1.005040e+10,11923969024,221891952640,226930999296,1.405930e+11,9.247508e+10,1.743462e+10,476082634752
142,expAE05,VUW_PISM2_s3,/Volumes/CrucialX8/computing/data/antarctica/i...,7234918912,79586549760,15014815744,12166726656,86025715712,13364265984,7154628608,...,20582791168,4971957760,1.088624e+10,11234804736,200028749824,225796849664,1.206116e+11,9.295081e+10,1.216673e+10,452918706176


In [65]:
22817908736*1.0

22817908736.0

In [66]:
2.281791e+10


22817910000.0

In [77]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/thermalforcing/computed_thermalforcing_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'


    #thermal foring 
#     d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )

    
    ind = np.where((d.time.values >=2270) & (d.time.values <= 2300))
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d2[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]=  yais
   

    for j in range(1,19):
        y =d2[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        
    for j in range(1,4):
        y =d2[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        for j,sector in enumerate(sectors):
            ys[j,:] = d2[f'{variable2}{sector}'].values[:tmin].copy()
        sumy = np.sum(ys.copy(),axis = 0)
        df[key].iloc[i]= np.nanmean(sumy[ind]).copy()
        
        

        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/{variable2}_2300.csv', index=False)


/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/205797514.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/205797514.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = y[ind].mean()
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/205797514.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-v